In [1]:
# Experimental Notebook using Keras' Hyperparameter Finetuning Library


In [3]:
# Inline plotting for notebooks
%matplotlib inline

# System and path setup
import sys
import os
from glob import glob
from copy import deepcopy
sys.path.append("../Al_data")

# Standard libraries
import datetime
import random
import pickle as pkl

# Scientific computing
import numpy as np
import pandas as pd
from scipy import stats

# Visualization
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Matminer
from matminer.featurizers.function import FunctionFeaturizer

# Scikit-learn
from sklearn.dummy import DummyRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import (
    train_test_split, cross_validate, cross_val_predict, 
    RepeatedKFold, GridSearchCV
)
from sklearn.linear_model import BayesianRidge, LinearRegression, Lasso, LassoLars
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, GlobalAveragePooling1D, 
    LayerNormalization, MultiHeadAttention, Embedding, Add, Layer
)

# Project-specific utilities
from stopping_power_ml.utils.io import load_qbox_data

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Activation, Bidirectional, Normalization
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import numpy as np
import random

In [4]:
import warnings; warnings.simplefilter('ignore')

## Load in the Dataset
This was created by a different notebook

In [5]:
data = pd.read_pickle(os.path.join('data', 'random_data_8_offset.pkl.gz'))
print('Data set size:', len(data))
data.head()

Data set size: 9800


,frame_id,force,position,velocity,energy,file_id,file,timestep,displacement,directory,...,"charge density at offset (0.00, 0.00, 0.00)","charge density at offset (0.00, 0.00, 0.10)","charge density at offset (-0.05, -0.05, 0.07)","charge density at offset (0.01, 0.09, 0.04)","charge density at offset (0.06, -0.08, 0.01)","charge density at offset (-0.10, 0.02, -0.01)","charge density at offset (0.08, 0.05, -0.04)","charge density at offset (-0.02, -0.07, -0.07)","charge density at offset (-0.00, 0.00, -0.10)",initial
14700,0,-0.000129,"[0.0, 5.74196597, 5.74196597]",1.0,-18651.709649,1,datasets/256_Al/Dv1.0,0,0.000000,datasets/256_Al/Dv1.0,...,-3.500000,-3.500783,-3.502303,-3.500028,-3.501622,-3.501313,-3.502013,-3.499906,-3.500783,True
14701,1,0.028183,"[0.00331934, 5.75126463, 5.75228826]",1.0,-18651.701982,1,datasets/256_Al/Dv1.0,1,0.014284,datasets/256_Al/Dv1.0,...,-3.500160,-3.500680,-3.502209,-3.499812,-3.501810,-3.501236,-3.501989,-3.500077,-3.501020,True
14702,2,0.054346,"[0.00663869, 5.76056328, 5.76261055]",1.0,-18651.701392,1,datasets/256_Al/Dv1.0,2,0.028568,datasets/256_Al/Dv1.0,...,-3.500266,-3.500522,-3.502066,-3.499542,-3.501950,-3.501200,-3.501917,-3.500194,-3.501209,True
14703,3,0.080006,"[0.00995803, 5.76986194, 5.77293284]",1.0,-18651.700428,1,datasets/256_Al/Dv1.0,3,0.042852,datasets/256_Al/Dv1.0,...,-3.500318,-3.500310,-3.501875,-3.499218,-3.502041,-3.501169,-3.501796,-3.500256,-3.501348,True
14704,4,0.104996,"[0.01327738, 5.77916059, 5.78325513]",1.0,-18651.699098,1,datasets/256_Al/Dv1.0,4,0.057136,datasets/256_Al/Dv1.0,...,-3.500315,-3.500043,-3.501635,-3.498841,-3.502084,-3.501083,-3.501628,-3.500264,-3.501439,True


Remove the initial transient

In [6]:
data.query('initial == False', inplace=True)
print('Training set size:', len(data))

Training set size: 9379


Determine which columns are inputs

In [7]:
featurizers = pkl.load(open(os.path.join('..', 'featurizers.pkl'), 'rb'))

In [8]:
X_cols = featurizers.feature_labels()
X_cols

['charge density at offset (0.00, 0.00, 0.00)',
 'charge density at offset (0.00, 0.00, 0.10)',
 'charge density at offset (-0.05, -0.05, 0.07)',
 'charge density at offset (0.01, 0.09, 0.04)',
 'charge density at offset (0.06, -0.08, 0.01)',
 'charge density at offset (-0.10, 0.02, -0.01)',
 'charge density at offset (0.08, 0.05, -0.04)',
 'charge density at offset (-0.02, -0.07, -0.07)',
 'charge density at offset (-0.00, 0.00, -0.10)',
 'AGNI_x in Al eta=8.00e-01',
 'AGNI_y in Al eta=8.00e-01',
 'AGNI_z in Al eta=8.00e-01',
 'AGNI_x in Al eta=1.23e+00',
 'AGNI_y in Al eta=1.23e+00',
 'AGNI_z in Al eta=1.23e+00',
 'AGNI_x in Al eta=1.88e+00',
 'AGNI_y in Al eta=1.88e+00',
 'AGNI_z in Al eta=1.88e+00',
 'AGNI_x in Al eta=2.89e+00',
 'AGNI_y in Al eta=2.89e+00',
 'AGNI_z in Al eta=2.89e+00',
 'AGNI_x in Al eta=4.43e+00',
 'AGNI_y in Al eta=4.43e+00',
 'AGNI_z in Al eta=4.43e+00',
 'AGNI_x in Al eta=6.80e+00',
 'AGNI_y in Al eta=6.80e+00',
 'AGNI_z in Al eta=6.80e+00',
 'AGNI_x in Al et

Determine which column is target

In [9]:
y_col = 'force'
print(y_col)

force


## A quick look at auto correlation
To determine if we should use a time-series optimized model such as convolutional or recurrent neural neworks, we see if energy and force (the two possible targets to predict) are correlated with themselves over time. We hypothesize that they are.


In [10]:
force_autocorr = data['force'].autocorr(lag=1)
print(f'Autocorrelation of force (lag=1): {force_autocorr:.4f}')

Autocorrelation of force (lag=1): 0.9964


In [11]:
energy_autocorr = data['energy'].autocorr(lag=1)
print(f'Autocorrelation of energy (lag=1): {energy_autocorr:.4f}')

Autocorrelation of energy (lag=1): 1.0000


This indeed shows a very strong correlation for force and a perfect correlation for energy. We will pursue models which work best for time-series data.

## Prepare train and validation split for Neural Networks

This section prepares the input data (`X`) and target labels (`y`) for training a recurrent neural network (RNN). A **sliding window** is applied to extract sequences of input features.


In [12]:
# Scale Features
x_scaler = StandardScaler()
y_scaler = StandardScaler()

data[X_cols] = x_scaler.fit_transform(data[X_cols])
data[y_col] = y_scaler.fit_transform(data[[y_col]])

In [13]:
sequence_col = 'file_id'  # column to group sequences by
timesteps = 20  # number of past steps used to predict the future

sequences = []  # list to store input sequences
targets = []  # list to store target values

for _, group in data.groupby(sequence_col):  # group data by sequence ID
    group = group.sort_values('timestep')  # ensure time order within each group
    x = group[X_cols].values  # extract input features
    y = group[y_col].values  # extract target values

    for i in range(len(group) - timesteps):  # slide window over each group
        sequences.append(x[i:i+timesteps])  # collect input sequence
        targets.append(y[i+timesteps])  # collect target after the sequence

X = np.array(sequences)        # shape: (samples, timesteps, features)
y = np.array(targets).reshape(-1, 1)  # reshape to 2D for scaler

print(X.shape)

(9259, 20, 36)


In [14]:
# Step 1: Split into train/validation (70/30 split)
num_samples = len(X)
split_index = int(0.5 * num_samples)

X_train, y_train = X[:split_index], y[:split_index]
X_val, y_val = X[split_index:], y[split_index:]

# Step 2: Create tf.data.Dataset objects
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))

# Step 3: Prepare datasets
batch_size = 32

train_ds = (
    train_ds
    .shuffle(buffer_size=len(X_train))   # Shuffle only train set
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

2025-07-11 14:14:21.074889: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [15]:
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1D
import kerastuner as kt

# Assuming TimePositionalEmbedding and TransformerEncoder are defined elsewhere

def build_transformer_model(hp):
    # Use the actual input shape here — do NOT hardcode
    input_shape = X_train.shape[1:]  # e.g. (20, 36)
    inputs = Input(shape=input_shape)
    
    embed_dim = hp.Int('embed_dim', 8, 64, step=8)
    num_heads = hp.Int('num_heads', 1, 4, step=1)
    ff_dim = hp.Int('ff_dim', 16, 128, step=16)
    dropout_rate = hp.Float('dropout_rate', 0.1, 0.5, step=0.1)
    use_positional_embedding = hp.Boolean('use_positional_embedding')
    
    x = Dense(embed_dim)(inputs)
    if use_positional_embedding:
        x = TimePositionalEmbedding(sequence_length=input_shape[0], embed_dim=embed_dim)(x)
    x = TransformerEncoder(embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model



In [16]:
class TransformerEncoder(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)
    
# === Positional Embedding Layer ===
class TimePositionalEmbedding(Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.pos_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_encoding = self.pos_embedding(positions)
        return x + pos_encoding

In [17]:
tuner = kt.RandomSearch(
    build_transformer_model,
    objective='val_loss',
    max_trials=40,
    executions_per_trial=1,
    directory='my_dir',
    project_name='transformer_tuning'
)


tuner.search(X_train, y_train,
             epochs=70,
             validation_data=(X_val, y_val),
             batch_size=32)

# Retrieve the best model
best_model = tuner.get_best_models(num_models=1)[0]
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_hp.values

Reloading Tuner from my_dir/transformer_tuning/tuner0.json


{'embed_dim': 48,
 'num_heads': 3,
 'ff_dim': 64,
 'dropout_rate': 0.30000000000000004,
 'use_positional_embedding': True}